# Лекция 08. Исключения и контекстные менеджеры

Программа читает небольшой отчёт. В одной строке вместо количества написано слово, другая строка обрывается посередине, а самого файла иногда вообще нет. Разберём, как понять причину сбоя, где принять решение о дальнейшей работе и как закрыть файл при любом обычном выходе из обработки.

## Цели

После лекции вы сможете:

- сравнивать специальное значение, статус, `errno` и исключение как способы сообщить о неудаче;
- читать traceback и находить место возникновения исключения;
- объяснять, как исключение проходит через вызовы функций;
- перехватывать ожидаемые ошибки, сохраняя видимость ошибок программы;
- проверять условия задачи через `raise`, сохранять явную причину через `raise ... from ...`, различать `except`, `else` и `finally`;
- открывать текстовые файлы с нужной кодировкой и управлять их временем жизни через `with`;
- различать освобождение ресурса и отмену уже сделанных изменений.

## Перед началом

Нужны функции, условия, циклы, строки, списки и словари из первых семи занятий. Все примеры работают на Python 3.14 со стандартной библиотекой, без интернета.

Файлы для чтения находятся в каталоге [examples](examples/). Откройте ноутбук из корня репозитория или из каталога `lesson08`. Результаты записи будем создавать во временных каталогах: учебные данные останутся прежними.

Сегодня пользуемся готовыми контекстными менеджерами. Устройство собственного класса-менеджера вернётся в теме ООП; генераторы и `@contextmanager` в эту лекцию не входят.

## Операция не смогла выполнить обещанное

Мы вызываем функцию, чтобы получить результат: найти цену, прочитать файл, разобрать количество. Следующие команды написаны в расчёте на то, что результат получен и им можно пользоваться. Но что делать самой функции, если выполнить обещанное не удалось?

Нужен способ сообщить вызывающему коду: **«Успешного результата нет. Продолжать первоначальный план нельзя»**. В разных языках и библиотеках это устраивают по-разному.

| Способ | Как сообщают о неудаче | Что должен сделать вызывающий код |
| --- | --- | --- |
| Специальное возвращаемое значение | Например, `None` или `-1` вместо результата | Проверить значение перед использованием |
| Статус отдельно от результата | Например, пара `(False, None)`; в других API — код возврата и результат в переданной структуре | Проверить статус и выбрать другой путь |
| Дополнительный канал ошибки | Возврат сигнализирует о неудаче, а подробности записаны, например, в `errno` | Проверить возврат, затем прочитать причину |
| Исключение | Обычного возврата результата не происходит | Для продолжения работы нужен подходящий обработчик |

Первые три способа действительно сообщают о неудаче. Однако сам по себе обычный возврат **не мешает выполнить следующую команду**. Проверку должен написать программист. Если её забыли, сигнал можно принять за данные и пойти дальше.

## Сигнал вернулся, а программа продолжила работу

В строке `codes` перечислены односимвольные коды товаров, а в списке `prices` — цены в том же порядке. Метод `str.find()` возвращает позицию найденной подстроки или `-1`, если её нет.

**В примере намеренно пропущена проверка результата поиска.** Кода `D` нет, но `-1` для списка означает последний элемент. Поэтому программа без единого исключения выдаст цену другого товара.

In [ ]:
codes = "ABC"
prices = [120, 80, 55]

found_position = codes.find("D")
wrong_price = prices[found_position]
print("Результат поиска:", found_position)
print("Цена товара D:", wrong_price)

assert found_position == -1
assert wrong_price == 55  # Это цена C. Программа выдала неверный результат.

Иногда продолжение приводит к сбою, но уже в другом месте. `dict.get()` без указанного значения по умолчанию возвращает `None`, когда ключа нет. Получить `None` можно без исключения; арифметика с ним сломается позже.

In [ ]:
stock = {"чай": 3}
missing_quantity = stock.get("соль")
print("Результат поиска:", missing_quantity)
print("Поиск закончен. Продолжаем собирать отчёт.")
assert missing_quantity is None

Следующая операция `missing_quantity + 2` уже поднимет `TypeError`. Сообщение будет об арифметике с `None`, хотя нужного результата не оказалось ещё при поиске в словаре. Между этими местами в настоящей программе может быть много вызовов.

Для таких интерфейсов проверка входит в договор использования: `found_position == -1`, `missing_quantity is None`, `if not success` для отдельного флага успеха. В промежуточных функциях отказ тоже приходится передавать дальше явно. Чем дальше место проверки от вызова, тем легче потерять первоначальную причину.

## Код возврата и `errno`

В интерфейсах C/POSIX часто разделяют сам факт неудачи и её причину. Например, `open()` при успехе возвращает неотрицательный файловый дескриптор — числовой идентификатор открытого файла. При неудаче он возвращает `-1`, а код причины находится в `errno`: например, `ENOENT` для отсутствующего файла или компонента пути, `EACCES` для отказа в доступе.

Ниже фрагмент C внутри функции. Здесь два шага: заметить неудачный возврат и прочитать причину. Продолжение успешного сценария прекращает написанная нами ветка с `return`.

```c
int fd = open("report.txt", O_RDONLY);
if (fd == -1) {
    int reason = errno;
    /* Здесь сообщаем об ошибке или выбираем другой сценарий. */
    return 1;
}
/* Здесь можно работать с открытым файлом. */
```

`errno` читают по правилам конкретной операции, после возврата, означающего ошибку; в примере — сразу после `fd == -1`. Успешный вызов не обязан обнулять `errno`, а другой вызов может изменить его. Поэтому одного условия «`errno` ненулевой» недостаточно, чтобы объявить последний вызов неудачным.

Если ветку `if` забыть, заполнение `errno` само не прервёт работу: следующая команда всё равно попытается использовать `fd`. Она тоже может вернуть признак ошибки, который снова придётся проверить.

## Исключение меняет обычный ход выполнения

У строк есть похожий метод `index()`: он тоже ищет позицию, но при отсутствии подстроки поднимает `ValueError`.

**Намеренно ошибочный пример** — отдельно его можно запустить, чтобы увидеть остановку:

```python
found_position = codes.index("D")  # Здесь возникает ValueError.
price = prices[found_position]      # До этой команды выполнение не дойдёт.
print("Цена товара D:", price)      # Неверная цена не будет напечатана.
```

Код проверки мы не написали, но успешный сценарий всё равно прервался **в месте неудачного поиска**. Операция не вернула обычный результат. Python начинает искать обработчик исключения и выходит из вызовов, в которых его нет.

![Обычный возврат позволяет продолжить выполнение, даже если сигнал ошибки проигнорирован; исключение переводит выполнение к поиску обработчика.](assets/error-signals.svg)

При выходе из вызовов выполняются завершающие действия — их разберём в блоках про `finally` и `with`. Если подходящего обработчика нигде не оказалось, выполнение скрипта завершается с ошибкой; в ноутбуке останавливается текущая ячейка.

**Разница в поведении по умолчанию:** при специальном значении или статусе нужно явно написать проверку, чтобы остановить ошибочный сценарий. При исключении нужно явно написать обработку, чтобы после неудачи продолжить работу осмысленно.

Если отсутствие товара — нормальный исход поиска, вернуть `None` удобно. Важно понимать договор операции и то, как он влияет на дальнейшее выполнение. Теперь посмотрим, как неудача проходит через несколько вызовов при чтении отчёта.

## Отчёт, который вчера читался

В строке два поля: название товара и целое количество, разделённые `;`. Сначала напишем прямой разбор без проверок.

На хорошем входе всё работает. Однако успешный запуск на трёх строках ещё не означает, что функция умеет работать с любым входом.

In [ ]:
def parse_item_basic(line: str) -> tuple[str, int]:
    name, raw_quantity = line.strip().split(";")
    return name.strip(), int(raw_quantity)


def total_quantity(lines: list[str]) -> int:
    total = 0
    for line in lines:
        name, quantity = parse_item_basic(line)
        total += quantity
    return total


print(total_quantity(["чай;2", "хлеб;1", "чай;3"]))

## Читаем traceback снизу вверх

**Намеренно ошибочный вызов:** `total_quantity(["чай;2", "хлеб;много", "чай;3"])`.
Он остановит выполнение ячейки. Ниже сокращённый traceback; служебные имена ячеек и номера строк опущены.

```text
Traceback (most recent call last):
  ... in <module>
    total_quantity(["чай;2", "хлеб;много", "чай;3"])
  ... in total_quantity
    name, quantity = parse_item_basic(line)
  ... in parse_item_basic
    return name.strip(), int(raw_quantity)
ValueError: invalid literal for int() with base 10: 'много'
```

Последняя строка сообщает тип и подробности ошибки. Выше видно выражение, которое не выполнилось, затем — кто его вызвал. Так можно отличить причину от места, где мы только заметили сбой.

`int("много")` не вернул число. Поэтому `parse_item_basic` не вернула пару, а `total_quantity` не вернула частичную сумму. Выполнение прервалось на второй строке; третья ещё не обрабатывалась.

## Тип исключения сообщает, что произошло

Исключение — объект, который передаёт сведения о прерывании обычного выполнения. У него есть тип и сообщение. Пока используем готовые типы Python.

| Тип | Пример причины |
| --- | --- |
| `ValueError` | `int("много")`: аргумент подходящего типа, но недопустимого значения |
| `TypeError` | `"2" + 2`: операция не поддерживает такое сочетание типов |
| `KeyError` | В словаре нет запрошенного ключа |
| `IndexError` | Индекс выходит за границы списка |
| `ZeroDivisionError` | Деление на ноль |
| `FileNotFoundError` | Не найден файл или один из каталогов пути |
| `PermissionError` | Операционная система отказала в доступе |

Сообщение помогает человеку разобраться, а в `except` выбирают **тип**. Сравнивать текст стандартной ошибки со строкой ненадёжно: формулировки могут отличаться.

У синтаксической ошибки другой момент возникновения: код не удалось разобрать ещё до выполнения. `try` внутри той же ошибочной ячейки не исправит пропущенное двоеточие.

## `try` задаёт участок, `except` — реакцию на ошибку

Если внутри `try` возникает подходящее исключение, оставшиеся команды этого блока пропускаются. Выполняется обработчик, затем работа продолжается **после всей конструкции**. Python не возвращается к строке, которая упала.

`as error` даёт имя объекту исключения. `str(error)` возвращает его сообщение.

In [ ]:
try:
    quantity = int("много")
    print("Эта строка не выполнится")
except ValueError as error:
    print("Количество не разобрано:", error)

print("Обработчик закончен, продолжаем работу")

## Ошибка может выйти из функции

Перехват не обязан находиться рядом с местом сбоя. Python ищет подходящий обработчик в текущем вызове, затем у вызывающей функции и дальше по цепочке. Если его нигде нет, исключение остаётся необработанным.

![Ошибка возникает в int, выходит из разбора строки и подсчёта и достигает обработчика вызывающего кода.](assets/exception-propagation.svg)

Функции разбора достаточно сообщить, что строка не подходит. Код, который знает задачу целиком, решает, попросить другой файл, пропустить строку или остановить импорт.

In [ ]:
try:
    total_quantity(["чай;2", "хлеб;много", "чай;3"])
except ValueError:
    print("Отчёт не удалось посчитать целиком")

assert total_quantity(["чай;2", "хлеб;0"]) == 2

## `raise`: правила задачи тоже могут нарушаться

Число `-3` успешно преобразуется через `int`, но отрицательное количество не подходит нашему отчёту. Проверим отдельно форму записи и ограничения задачи.

Договоримся: полей ровно два, название непустое, количество — целое неотрицательное число. Пробелы вокруг полей разрешены. Допустимую запись целого числа определяет `int()`. Нулевое количество корректно.

При нарушении договора функция поднимает `ValueError`. Она не печатает сообщение и не подставляет ноль: решение о дальнейшей работе остаётся у вызывающего кода.

In [ ]:
def parse_item(line: str) -> tuple[str, int]:
    fields = line.strip().split(";")
    if len(fields) != 2:
        raise ValueError("Ожидались два поля: название;количество")

    name, raw_quantity = fields
    name = name.strip()
    if not name:
        raise ValueError("Название не должно быть пустым")

    quantity = int(raw_quantity)
    if quantity < 0:
        raise ValueError("Количество должно быть неотрицательным")
    return name, quantity


assert parse_item(" чай ; 0 ") == ("чай", 0)
for line in ["хлеб;много", "мыло;-1", "без разделителя"]:
    try:
        parse_item(line)
    except ValueError as error:
        print(repr(line), "→", error)

## Перехватываем там, где есть решение

В нашем примере неверная строка — ожидаемая проблема данных. Ошибка в самом алгоритме, например обращение к несуществующей переменной, требует исправления программы.

`except Exception:` перехватывает очень широкую группу ошибок, включая многие ошибки кода. Голый `except:` ещё шире: он может перехватить и остановку программы через `Ctrl+C`. Для разбора количества такие обработчики слишком широки.

Даже `ValueError` не стоит ловить вокруг всей программы. Оставим в `try` только действие, от которого ожидаем именно эту ошибку. Обновление итогов будет за его пределами.

Заранее известное правило удобно проверять через `if`: например, запрещённое отрицательное количество. Ошибку библиотечного преобразования удобно обработать через `except`. Обе конструкции нужны.

## Одна плохая строка или весь отчёт?

Выберем политику для предварительного просмотра: корректные строки попадают в свод, некорректные — в отдельный список с номером и причиной. Ошибки не исчезают: пользователь видит, какие записи не учтены.

Номер считаем по исходным строкам, начиная с 1. В этом формате **пустая строка тоже считается некорректной**. Повторные названия суммируем, нулевые количества сохраняем.

Блок `else` выполняется, когда `try` завершился без исключения. Ошибка из самого `else` уже не будет поймана соседним `except`.

In [ ]:
def collect_items(lines):
    totals = {}
    rejected = []
    for number, line in enumerate(lines, start=1):
        try:
            name, quantity = parse_item(line)
        except ValueError as error:
            rejected.append((number, str(error)))
        else:
            totals[name] = totals.get(name, 0) + quantity
    return totals, rejected


lines = ["чай;2", "хлеб;много", "чай;3", "мыло;-1", "хлеб;0"]
totals, rejected = collect_items(lines)
assert totals == {"чай": 5, "хлеб": 0}
assert [number for number, message in rejected] == [2, 4]
print("Учтено:", totals)
print("Отклонено:", rejected)

## Обработчик может передать исключение дальше

Иногда в обработчике нужно добавить запись в журнал, а решение оставить внешнему коду. `raise` без аргумента повторно поднимает текущее исключение.

Ниже выбран другой договор: отчёт должен быть корректен целиком. Сообщаем номер проблемной строки и прерываем обработку. В демонстрации журналом служит `print`; функция не возвращает частичный результат.

In [ ]:
def strict_total(lines: list[str]) -> int:
    total = 0
    for number, line in enumerate(lines, start=1):
        try:
            name, quantity = parse_item(line)
        except ValueError:
            print(f"Не разобрана строка {number}")
            raise
        total += quantity
    return total


try:
    strict_total(["чай;2", "хлеб;много", "чай;3"])
except ValueError as error:
    print("Импорт остановлен:", error)

## `raise ... from ...`: сообщить о задаче и сохранить причину

`int()` знает, что текст `"много"` не удалось преобразовать в число. Код чтения отчёта знает больше: это вторая строка и её содержимое некорректно. Можно поднять новое исключение с сообщением на уровне отчёта и явно указать исходное исключение как причину.

Для этого используют `raise новое_исключение from исходное_исключение`. Вызывающий код получает новое исключение, а при диагностике видны и исходный сбой, и добавленные подробности.

В примере `parse_item` по-прежнему разбирает одну запись. Обёртка добавляет номер строки и сохраняет связь с причиной. Своих классов исключений для этого не нужно: пока используем `ValueError` и для разбора, и для ошибки записи.

In [ ]:
import traceback


def parse_report_row(line: str, number: int) -> tuple[str, int]:
    try:
        return parse_item(line)
    except ValueError as error:
        raise ValueError(
            f"Строка {number}: некорректная запись {line!r}"
        ) from error


try:
    parse_report_row("хлеб;много", 2)
except ValueError as error:
    print("".join(traceback.format_exception(error)), end="")

В выводе сначала идёт traceback первоначального `ValueError` от `int()`, затем фраза `The above exception was the direct cause of the following exception` — «предыдущее исключение стало непосредственной причиной следующего». После неё — traceback нового исключения с номером и текстом строки.

Здесь `except` нужен, чтобы показать цепочку и продолжить учебный ноутбук. `traceback.format_exception(error)` готовит её полный текст. Обычный `print(error)` вывел бы только сообщение нового исключения.

Сравним способы передачи ошибки внутри обработчика:

| Конструкция | Что передаём дальше и что увидим в traceback |
| --- | --- |
| `raise` | То же исключение с его traceback |
| `raise ValueError("...")` | Новое исключение; Python автоматически сохранит контекст ошибки, во время обработки которой оно возникло |
| `raise ValueError("...") from error` | Новое исключение с явно указанной причиной; traceback покажет причинную связь |
| `raise ValueError("...") from None` | Новое исключение; показ предшествующего контекста в стандартном traceback будет подавлен |

Обычный `raise ValueError(...)` внутри `except` тоже сохраняет исходную ошибку как контекст. Его traceback связывает события словами «во время обработки предыдущего исключения возникло следующее». Запись `from error` точнее выражает намерение: **новое исключение вызвано именно этой причиной**.

### Когда встречается `from None`

Иногда интерфейс намеренно показывает только подготовленное сообщение, скрывая подробности внутреннего преобразования. Тогда в обработчике пишут `from None`.

**Намеренно ошибочный пример для отдельного запуска:**

```python
try:
    int("много")
except ValueError:
    raise ValueError("Количество должно быть целым числом") from None
```

В стандартном traceback останется новое исключение с нашим сообщением; цепочка первоначального `ValueError` от `int()` показана не будет. Сам исходный контекст в объекте исключения сохраняется. `from None` управляет отображением, а ошибка продолжает распространяться.

Для диагностики отчёта номер строки и исходная причина полезны вместе, поэтому в `parse_report_row` выбран `from error`.

## Что именно означает «обработать ошибку»

| Решение | Когда оно соответствует задаче |
| --- | --- |
| Прервать обработку | Неполный результат нельзя использовать |
| Пропустить запись и сохранить причину | Допустим предварительный результат с явным списком потерь |
| Подставить значение по умолчанию | В договоре прямо сказано, что оно означает |
| Сообщить и повторно поднять исключение | Здесь известны детали, а решение принимает внешний код |

`except: pass` сам по себе не задаёт полезной политики. Особенно опасно молча считать ошибочное количество нулём: мы перестаём отличать реальный ноль от потерянных данных.

## У ресурса есть время жизни

Открытый файл связан с ресурсом операционной системы. Его нужно закрыть, когда работа закончена. Закрытие может также сбросить буфер записи; ожидание автоматической уборки объектов для этого не подходит.

Последовательность «открыть → прочитать → закрыть» ломается, если чтение или обработка поднимает исключение: до последней команды выполнение не дойдёт. Закрытие должно происходить и при успешном `return`, и при ошибке.

Сначала покажем механизм на `StringIO` — небольшом текстовом файле в памяти. У него есть `readline()`, `close()` и признак `closed`; отдельный файл на диске не нужен.

## `finally`: действие перед выходом

`finally` выполняется при обычном завершении `try`, при `return` и при выходе из-за исключения. После него незавершённый выход продолжается: возвращается результат или дальше распространяется ошибка.

![Три пути выхода из try — успех, return и исключение — проходят через finally.](assets/finally-flow.svg)

В примере функция принимает владение переданным потоком: после вызова он должен быть закрыт. Это часть её договора.

In [ ]:
from io import StringIO


def first_number_with_finally(source) -> int:
    try:
        return int(source.readline())
    finally:
        source.close()


good_source = StringIO("12\n8\n")
assert first_number_with_finally(good_source) == 12
assert good_source.closed

bad_source = StringIO("много\n")
try:
    first_number_with_finally(bad_source)
except ValueError as error:
    print("Ошибка сохранилась:", error)
assert bad_source.closed
print("Оба потока закрыты")

Освобождение ресурса и обработка ошибки — разные действия. `finally` закрыл поток, но не превратил неверный текст в число и не отменил исключение.

В `finally` помещают короткую завершающую работу. Если она сама поднимет исключение, наблюдаемая ошибка изменится. `return` внутри `finally` тоже может перекрыть результат или ошибку из `try`, поэтому возвращать результат оттуда не следует.

Гарантия касается нормальной работы интерпретатора: при аварии процесса, отключении питания или принудительном завершении ОС завершающий код может не выполниться.

## `with`: договор о входе и выходе

Контекстный менеджер определяет действия при входе в блок и при выходе из него. `with` вызывает их в нужные моменты. Для файла действие выхода — закрытие.

Перепишем функцию. Теперь завершением работы с потоком управляет `with`, а внутри осталось только чтение и преобразование. Для файлов это привычная замена ручному `try/finally` с `close()`.

Успешный вход в контекст гарантирует вызов действия выхода при обычном завершении блока, `return` или исключении. Сама ошибка не обязана исчезнуть: файловый менеджер её не подавляет.

In [ ]:
def first_number(source) -> int:
    with source:
        return int(source.readline())


good_source = StringIO("12\n8\n")
assert first_number(good_source) == 12
assert good_source.closed

bad_source = StringIO("много\n")
try:
    first_number(bad_source)
except ValueError:
    print("Ошибка вышла из with, поток уже закрыт")
assert bad_source.closed

![Файл открыт только во время блока with; при успешном завершении, return и исключении выполняется закрытие.](assets/file-lifetime.svg)

У разных менеджеров действие выхода разное: один закрывает ресурс, другой восстанавливает временное состояние. Само слово `with` не означает «закрыть всё» или «поймать все исключения».

Не всякий объект можно использовать после `with`. Файловый объект остаётся доступен по имени, но чтение закрытого файла уже невозможно.

## Теперь настоящий файл

Файл [examples/items.txt](examples/items.txt) содержит те же товары и несколько испорченных строк. `open` возвращает файловый объект. В `as source` мы даём этому объекту имя.

`encoding="utf-8"` явно задаёт кодировку наших учебных файлов. Открытый текстовый файл можно обходить обычным `for`: он выдаёт строки по одной. Этого знания сейчас достаточно; устройство итераторов разберём отдельно.

In [ ]:
from pathlib import Path

lesson_dir = Path("lesson08") if Path("lesson08").is_dir() else Path(".")
data_path = lesson_dir / "examples" / "items.txt"

with open(data_path, encoding="utf-8") as source:
    totals, rejected = collect_items(source)

assert source.closed
assert totals == {"чай": 5, "хлеб": 0}
assert [number for number, message in rejected] == [2, 4, 6]
print("Учтено:", totals)
print("Номера отклонённых строк:", [number for number, message in rejected])

Чтобы прочитать весь небольшой файл, используют `source.read()`. `source.readline()` читает одну строку, а `for line in source` последовательно обрабатывает строки до конца файла. Перевод строки обычно остаётся в прочитанном тексте; наш разбор убирает пробелы по краям через `strip()`.

| Режим | Что происходит при открытии |
| --- | --- |
| `"r"`, по умолчанию | Чтение существующего файла |
| `"w"` | Создание файла или **сразу очистка** существующего |
| `"a"` | Запись в конец файла; при отсутствии файл создаётся |

В текстовый файл записывают строки через `write()`. Переводы строк нужно добавлять самим. Число для записи сначала превращают в строку, например через f-строку.

## Ошибка строки и ошибка доступа — разные проблемы

`collect_items` ловит `ValueError` только от разбора **уже прочитанной строки**. Если файл не открылся или чтение не удалось, это не повод добавлять фиктивную строку в список отклонённых.

Вынесем чтение в функцию. Сама она не выбирает способ общения с человеком. Вызывающий код может отдельно отреагировать на отсутствие файла.

`FileNotFoundError` и `PermissionError` входят в группу `OSError`. Обработчик `except OSError` подходит для общей реакции на сбой файловой операции, но сообщение «файл не найден» будет верным только для `FileNotFoundError`. Если нужны оба обработчика, более конкретный ставят первым: срабатывает первый подходящий.

In [ ]:
from tempfile import TemporaryDirectory


def read_inventory(path):
    with open(path, encoding="utf-8") as source:
        return collect_items(source)


with TemporaryDirectory() as directory:
    missing_path = Path(directory) / "нет-отчёта.txt"
    try:
        read_inventory(missing_path)
    except FileNotFoundError:
        print("Выберите существующий файл отчёта")

assert read_inventory(data_path)[0] == {"чай": 5, "хлеб": 0}

## Два ресурса: читаем и записываем

В одном `with` можно перечислить несколько менеджеров. Они входят слева направо, а завершают работу в обратном порядке. Если вход во второй контекст не удался, первый уже вошедший контекст будет завершён.

Запишем корректный свод в отдельный временный файл. Вход открывается первым, выход — вторым. Пути различаются: открытие исходного файла в режиме `"w"` уничтожило бы его содержимое.

`TemporaryDirectory` тоже контекстный менеджер: при выходе удаляет созданный временный каталог вместе с его файлами. Поэтому читаем результат внутри внешнего блока, когда каталог ещё существует.

In [ ]:
with TemporaryDirectory() as directory:
    output_path = Path(directory) / "summary.txt"
    with (
        open(data_path, encoding="utf-8") as source,
        open(output_path, "w", encoding="utf-8") as target,
    ):
        totals, rejected = collect_items(source)
        for name, quantity in totals.items():
            target.write(f"{name};{quantity}\n")

    assert source.closed and target.closed
    result_text = output_path.read_text(encoding="utf-8")
    assert result_text == "чай;5\nхлеб;0\n"
    print(result_text, end="")

assert not output_path.exists()

## Неожиданно, но по правилам

### 1. Закрыть файл — не значит отменить запись

`with` гарантирует завершающее действие менеджера. Обычный файловый менеджер не умеет возвращать прежнее содержимое. Если программа записала часть результата и упала, записанная часть может остаться в файле.

Для задачи «получить новый файл целиком или сохранить старый» нужен отдельный способ записи: сначала подготовить результат во временном файле, затем заменить целевой. Сегодня такую операцию не реализуем.

In [ ]:
with TemporaryDirectory() as directory:
    output_path = Path(directory) / "partial.txt"
    try:
        with open(output_path, "w", encoding="utf-8") as target:
            target.write("Первая строка\n")
            raise ValueError("Не удалось подготовить вторую строку")
    except ValueError as error:
        print("Запись прервалась:", error)

    assert target.closed
    assert output_path.read_text(encoding="utf-8") == "Первая строка\n"
    print("В файле осталась первая строка")

### 2. Исключение не отменяет изменения списка или словаря

Если до сбоя мы добавили запись в список или изменили словарь, исключение само по себе не вернёт их назад. У `strict_total` частичный результат был локальным и не вышел наружу, но внешние изменения потребовали бы отдельного решения. Нужен ли принцип «всё или ничего», определяет задача.

### 3. Закрытый файл всё ещё можно хранить в переменной

Переменная `source` после блока остаётся, а ресурс закрыт. Проверка `source.closed` корректна; попытка `source.read()` приведёт к `ValueError`. Удалять имя для закрытия файла не требуется.

### 4. Обработчик не защищает сам себя

Если внутри `except` произойдёт новая ошибка, соседний обработчик той же конструкции её не поймает. Так же устроен `else`. Поэтому сообщение об ошибке и завершающие действия тоже должны быть простыми и надёжными.

## Итоги

Специальное значение, статус и `errno` требуют явной проверки результата вызова. Исключение прерывает успешный сценарий само; дальнейшую реакцию выбирает подходящий обработчик. Если его нет, выполнение скрипта или текущей ячейки завершается с ошибкой.

| Конструкция | За что отвечает |
| --- | --- |
| `try` | Участок, из которого ожидаем исключение |
| `except SomeError` | Реакция на первый подходящий тип ошибки |
| `else` | Работа после успешного завершения `try` |
| `finally` | Завершающее действие перед выходом |
| `raise ValueError(...)` | Сообщение о нарушенном договоре функции |
| `raise` в обработчике | Передача текущего исключения дальше |
| `raise ValueError(...) from error` | Новое исключение с явно указанной исходной причиной |
| `raise ValueError(...) from None` | Новое исключение без показа предшествующего контекста в стандартном traceback |
| `with` | Вход в контекст и гарантированный вызов действия выхода после успешного входа |

В примере с отчётом мы разделили три решения: разобрать строку, выбрать политику для некорректных данных и управлять временем жизни файла. Плохая запись не исчезла из отчёта, ошибка доступа не стала пустым результатом, а открытые файлы закрылись.

**Обсудим перед семинаром:** импорт должен принять только полностью корректный файл. В каком месте нужно прекратить ловить ошибки строк? Что можно безопасно вернуть вызывающему коду? Какие изменения сохранятся, если мы уже успели записать часть результата?

Дальше — [семинар](seminar.ipynb) с заготовками и проверками и [домашние задачи](tasks.md).

## Источники

- [Встроенные типы — Python 3.14](https://docs.python.org/3.14/library/stdtypes.html): поведение `str.find()`, `str.index()`, `dict.get()` и отрицательных индексов.
- [`open(2)` — Linux man-pages](https://man7.org/linux/man-pages/man2/open.2.html): файловый дескриптор при успехе, `-1` и `errno` при неудаче.
- [`errno(3)` — Linux man-pages](https://man7.org/linux/man-pages/man3/errno.3.html): связь кода возврата с `errno`, чтение причины после неудачного вызова.
- [Ошибки и исключения — Python 3.14](https://docs.python.org/3.14/tutorial/errors.html): `try`, `except`, `else`, `raise`, цепочки исключений и `finally`.
- [Контекст исключения — Python 3.14](https://docs.python.org/3.14/library/exceptions.html#exception-context): неявный контекст, явная причина и подавление показа контекста через `from None`.
- [Оператор `with` — справочник языка Python 3.14](https://docs.python.org/3.14/reference/compound_stmts.html#the-with-statement): вход и выход из контекста, несколько менеджеров.
- [Чтение и запись файлов — Python 3.14](https://docs.python.org/3.14/tutorial/inputoutput.html#reading-and-writing-files): режимы открытия, кодировка и файловые методы.

В документации могут встречаться темы следующих лекций: собственные классы исключений, протокол контекстного менеджера, группы исключений. Для текущего занятия они не требуются.

## Пример экзаменационного вопроса

**Тип:** что выведет код и почему.

**Вопрос.** Что выведет программа и почему? Объясните, что произойдёт с потоком и уже накопленной суммой после ошибки.

```python
from io import StringIO

source = StringIO("12\nмного\n")
total = 0

try:
    with source:
        total += int(source.readline())
        total += int(source.readline())
        print("готово")
except ValueError:
    print("ошибка")
finally:
    print("закрыт:", source.closed)

print("сумма:", total)
```

**Пример полного ответа.**

```text
ошибка
закрыт: True
сумма: 12
```

Первая строка превращается в число `12`, и `total` становится равен `12`. На второй строке `int()` поднимает `ValueError`: слово `много` нельзя преобразовать в целое число. Вычисление правой части не завершилось, поэтому второе прибавление не произошло. Команда `print("готово")` пропускается.

При выходе из `with` поток закрывается. Контекстный менеджер `StringIO` не подавляет исключение, и оно доходит до `except ValueError`, который печатает `ошибка`. Затем выполняется `finally`: к этому моменту поток уже закрыт, поэтому `source.closed` равен `True`. Само закрытие выполнил менеджер, а `finally` здесь только вывел его результат.

Исключение обработано, поэтому программа продолжает работу после всей конструкции и печатает `сумма: 12`. Ошибка не откатывает уже выполненные изменения: первое прибавление сохраняется. Закрытый поток также остаётся доступен по имени `source`, хотя читать из него больше нельзя.

**Пример неполного ответа.**

«Будет напечатано `ошибка`, затем `закрыт: True` и `сумма: 12`. Строку `много` нельзя преобразовать в `int`, поэтому сработает `except ValueError`».

**Чего не хватает.** Вывод указан верно, но не объяснено, кто и когда закрывает поток, почему `with` не устраняет ошибку и почему после неё сохраняется первая прибавленная сумма. Для полного ответа нужно проследить выход из `with`, обработку исключения и продолжение работы программы.
